### 3.5 Inpatient Medication Titration (`change`) and Glycemic Instability

#### Monitoring Changes in Diabetes Treatment

Medication adjustments during a hospital stay can provide useful information about how a patient's diabetes is being managed. In this dataset, the `change` variable indicates whether a patient's diabetes medication was modified (`Ch`) or remained unchanged (`No`) during hospitalization.

The `insulin` variable provides additional details about insulin treatment, with four possible values: `Up`, `Down`, `Steady`, and `No`. These categories indicate whether insulin dosage was increased, decreased, maintained at the same level, or not recorded as being used.

This analysis compares 30 day readmission rates across these medication adjustment categories to examine whether patients who experienced dosage changes had different readmission patterns from those whose treatment remained unchanged. Particular attention is given to upward and downward insulin adjustments, as these may reflect changes in the patient's treatment requirements during the hospital stay.

The main objective is to determine whether inpatient medication adjustments are associated with an increased likelihood of returning to the hospital within 30 days. The findings may provide insight into the relationship between diabetes management during hospitalization and patient outcomes after discharge.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

chg_regimen = df_cleaned.groupby(['Medication_Cohort', 'change'])['readmitted_30d'].mean().mul(100).reset_index()
chg_regimen['Dosage_Status'] = chg_regimen['change'].map({'Ch': 'Dosage Changed (Ch)', 'No': 'Unchanged (No)'})

sns.barplot(data=chg_regimen, x='Medication_Cohort', y='readmitted_30d', hue='Dosage_Status', palette=['#e67e22', '#34495e'], ax=ax1)
ax1.set_title("Readmission Rate by Regimen & Dosage Change", fontweight='bold', fontsize=12)
ax1.set_xlabel("Medication Cohort", fontweight='bold')
ax1.set_ylabel("30-Day Readmission Rate (%)", fontweight='bold')
ax1.tick_params(axis='x', rotation=15)
for p in ax1.patches:
    if p.get_height() > 0:
        ax1.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width()/2., p.get_height() + 0.2),
                     ha='center', fontsize=8.5, fontweight='bold')


ins_order = ['No', 'Steady', 'Up', 'Down']
ins_rates = df_cleaned.groupby('insulin')['readmitted_30d'].mean().mul(100).reindex(ins_order).reset_index()

sns.barplot(data=ins_rates, x='insulin', y='readmitted_30d', palette='Reds', ax=ax2, edgecolor='black')
ax2.set_title("Impact of Insulin Titration Direction on 30-Day Return", fontweight='bold', fontsize=12)
ax2.set_xlabel("Inpatient Insulin Dosage Status", fontweight='bold')
ax2.set_ylabel("30-Day Readmission Rate (%)", fontweight='bold')
ax2.axhline(df_cleaned['readmitted_30d'].mean() * 100, color='black', linestyle='--', label='Network Avg')
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width()/2., p.get_height() + 0.3),
                 ha='center', fontweight='bold', fontsize=10)
ax2.legend()

plt.tight_layout()
plt.show()

### 3.6 Multicollinearity Analysis of Numerical Clinical Features

#### Examining Relationships Between Clinical Variables

Several numerical variables in the dataset describe how patients use hospital services during their treatment. However, some of these variables may be related to one another. For example, patients who stay longer in the hospital may also receive more medications or undergo additional laboratory tests.

Before using these variables in further analysis or predictive modelling, it is important to examine the relationships between them. This analysis considers eight numerical features: `time_in_hospital`, `num_lab_procedures`, `num_procedures`, `num_medications`, `number_outpatient`, `number_emergency`, `number_inpatient`, and `number_diagnoses`.

A Pearson correlation matrix is used to measure the strength and direction of the linear relationships between these variables. The results are presented using a lower-triangle heatmap, which displays each pairwise correlation without repeating the same information on both sides of the matrix.

Particular attention is given to possible relationships between the number of medications and hospital stay duration, as well as the connections between previous outpatient, emergency, and inpatient visits. The analysis also examines how historical hospital utilization relates to the 30-day readmission outcome.

Understanding these relationships helps identify numerical features that may contain overlapping information and provides useful insight before moving towards more advanced statistical analysis and predictive modelling.

In [ ]:
num_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'readmitted_30d'
]

corr_matrix = df_cleaned[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(10, 7.5))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='vlag', vmin=-0.2, vmax=0.5, center=0,
    square=True, linewidths=1.2, cbar_kws={"shrink": 0.8, "label": "Pearson Correlation (r)"}
)
plt.title("Lower-Triangle Multicollinearity Matrix of Numerical Clinical Features", fontweight='bold', fontsize=13)
plt.xticks(rotation=35, ha='right', fontweight='bold')
plt.yticks(fontweight='bold')
plt.tight_layout()
plt.show()

### 3.7 Relationship Between Length of Stay and Laboratory Procedures

#### Investigating Hospital Stay and Testing Activity

This analysis examines whether the number of laboratory procedures performed during a hospital stay tends to increase with the length of that stay. The two variables considered are `time_in_hospital` and `num_lab_procedures`. The main objective is to identify whether a positive linear relationship exists between them.

Two charts are used to examine this relationship from different perspectives:

- **Left Chart  – Bar Chart:** Displays the average number of laboratory procedures performed for each hospital stay duration, covering stays from 1 to 14 days. This makes it easier to compare laboratory activity across different lengths of stay.

- **Right Chart – Scatter Plot with Regression Trendline:** Uses a randomly selected sample of 1,000 patient encounters to display the relationship between hospital stay duration and the number of laboratory procedures. A red regression line is added to highlight the overall direction of the relationship.

Together, these charts provide a clearer view of how laboratory testing activity varies with hospital stay duration. They also help determine whether longer hospital stays are generally associated with a greater number of laboratory procedures.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

avg_labs = df_cleaned.groupby('time_in_hospital')['num_lab_procedures'].mean().reset_index()

sns.barplot(data=avg_labs, x='time_in_hospital', y='num_lab_procedures', color='#3498db', ax=ax1)
ax1.set_title("Average Lab Procedures by Hospital Days", fontweight='bold')
ax1.set_xlabel("Time in Hospital (Days)")
ax1.set_ylabel("Average Number of Lab Tests")

for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.0f}", 
                 (p.get_x() + p.get_width() / 2., p.get_height() + 0.8),
                 ha='center', fontsize=9, fontweight='bold')

sample_df = df_cleaned.sample(n=1000, random_state=42)

sns.regplot(data=sample_df, x='time_in_hospital', y='num_lab_procedures',
            scatter_kws={'alpha': 0.25, 'color': '#2c3e50'},
            line_kws={'color': 'red', 'linewidth': 2}, ax=ax2)

ax2.set_title("Linear Correlation: Hospital Stay vs. Lab Tests", fontweight='bold')
ax2.set_xlabel("Time in Hospital (Days)")
ax2.set_ylabel("Number of Lab Procedures")

plt.tight_layout()
plt.show()

corr_value = df_cleaned['time_in_hospital'].corr(df_cleaned['num_lab_procedures'])
print(f"Pearson Correlation (r): {corr_value:.2f} -> Positive Linear Correlation (Longer stays = More lab tests)")

### 3.8 Initial Length of Stay Distribution and Post-Acute Discharge Disposition

#### Hospital Stay Duration and Discharge Decisions

This part of the analysis focuses on two aspects of hospital care: the duration of a patient's initial hospital stay and the destination to which the patient is discharged. Both are examined to understand how they relate to readmission within 30 days.

**1. Comparing Initial Hospital Stay Duration**

The first analysis compares the length of hospital stays between patients who were readmitted within 30 days (`<30`) and those who were not readmitted (`NO`). A combined violin and boxplot is used to examine the distribution of `time_in_hospital` across these two groups.

The visualisation provides information about the median, interquartile range, overall distribution shape, and group mean. These measurements help determine whether patients who return to the hospital within 30 days tend to have different initial stay durations from those who are not readmitted.

**2. Examining Discharge Destinations**

The second analysis investigates whether readmission rates vary depending on where patients are sent after leaving the hospital. The `discharge_disposition_desc` variable is grouped into several main discharge categories to make the comparison easier.

The analysis focuses on four discharge pathways:

- **Home:** Patients discharged directly to their homes.
- **Skilled Nursing Facilities (SNF):** Patients transferred to facilities that provide nursing care and assistance with daily activities.
- **Home Health Services:** Patients discharged home with additional healthcare support.
- **Post-Acute/Rehabilitation Facilities:** Patients transferred to facilities where they can receive further treatment and rehabilitation.

The 30-day readmission rates are then compared across these categories to identify differences between discharge pathways. This may help highlight areas where additional follow-up care or better coordination between healthcare providers could be beneficial.

In [ ]:
def simplify_discharge_disp(desc):
    if pd.isna(desc): return 'Other Disposition'
    desc_str = str(desc).lower()
    if 'home' in desc_str and 'service' not in desc_str:
        return 'Discharged to Home'
    elif 'snf' in desc_str or 'skilled nursing' in desc_str:
        return 'Skilled Nursing Facility (SNF)'
    elif 'home health service' in desc_str:
        return 'Home with Health Service'
    elif 'short term' in desc_str or 'rehab' in desc_str or 'transfer' in desc_str:
        return 'Post-Acute / Rehab Transfer'
    else:
        return 'Other Disposition'

df_cleaned['Discharge_Group'] = df_cleaned['discharge_disposition_desc'].apply(simplify_discharge_disp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Left- Violin Plot with Embedded Boxplot for Length of Stay
df_sub = df_cleaned[df_cleaned['readmitted'].isin(['NO', '<30'])]
sns.violinplot(
    data=df_sub, x='readmitted', y='time_in_hospital', order=['NO', '<30'],
    palette=['#2ecc71', '#e74c3c'], inner='box', cut=0, ax=ax1
)
means = df_sub.groupby('readmitted')['time_in_hospital'].mean()[['NO', '<30']]
ax1.scatter([0, 1], means.values, color='yellow', edgecolor='black', s=90, zorder=5, label='Cohort Mean')
for i, m in enumerate(means.values):
    ax1.annotate(f"Mean: {m:.2f}d", (i, m + 0.6), ha='center', fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax1.set_title("Length of Stay Distribution: Non-Readmitted vs. <30 Days", fontweight='bold', fontsize=12)
ax1.set_xlabel("Readmission Outcome", fontweight='bold')
ax1.set_ylabel("Time in Hospital (Days)", fontweight='bold')
ax1.legend()

# Right- Horizontal Bar Chart of Discharge Disposition Risk
disp_summary = df_cleaned.groupby('Discharge_Group')['readmitted_30d'].agg(
    Encounters='count',
    Rate=lambda x: x.mean() * 100
).sort_values(by='Rate', ascending=True).reset_index()

bars = ax2.barh(disp_summary['Discharge_Group'], disp_summary['Rate'], color='#8e44ad', alpha=0.85, edgecolor='black')
ax2.axvline(df_cleaned['readmitted_30d'].mean() * 100, color='red', linestyle='--', linewidth=1.5, label='Network Average')
ax2.set_title("30-Day Readmission Rate by Discharge Pathway", fontweight='bold', fontsize=12)
ax2.set_xlabel("30-Day Readmission Rate (%)", fontweight='bold')
ax2.set_xlim(0, 22)  

for bar in bars:
    w = bar.get_width()
    ax2.annotate(f" {w:.2f}%", (w, bar.get_y() + bar.get_height()/2),
                 va='center', ha='left', fontweight='bold', fontsize=10)
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()